# Final Walkthrough: Capabilities and Usage

This notebook is a concise user-facing guide to the API that is already implemented in this repository.

The purpose is practical: declare fields and gauge groups, write Lagrangians with the supported objects, compile models, and extract momentum-space vertices. The emphasis is on usage rather than implementation details.

The workflow used throughout the notebook is:

1. declare fields and gauge groups,
2. write either a local `Lagrangian(...)` or a `Model(..., lagrangian_decl=...)`,
3. call `model.lagrangian()` when metadata-dependent terms such as `CovD(...)`, `FieldStrength(...)`, `GaugeFixing(...)`, or `GhostLagrangian(...)` are involved,
4. extract a chosen vertex with `feynman_rule(...)`, or enumerate available vertices with zero-argument `feynman_rule()`.

Only the implemented library API is used below. The notebook defines one small display helper, `show(...)`, so that repeated vertex output is easier to read; it is not part of the library API.


In [1]:
import sys
from pathlib import Path
from fractions import Fraction

root = Path.cwd()
if not (root / "src").exists():
    root = root.parent

src_path = root / "src"
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from symbolica import S, Expression
from symbolic.vertex_engine import I
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from model import (
    CovD,
    Field,
    FieldStrength,
    Gamma,
    GaugeFixing,
    GaugeGroup,
    GaugeRepresentation,
    GhostLagrangian,
    Lagrangian,
    Metric,
    Model,
    PartialD,
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
)

mu = S("mu")
nu = S("nu")

lam4 = S("lam4")
y = S("y")
gBox = S("gBox")
e = S("e")
gs = S("gs")
g1 = S("g1")
g2 = S("g2")
Qe = S("Qe")
YL = S("YL")
YR = S("YR")
YH = S("YH")
xi = S("xi")


def show(title, result):
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", expression)
            print()
    else:
        print(result)
        print()


print("Python:", sys.executable)
print("Repository root:", root.resolve())


Python: /Users/rems/Documents/MScThesis/.venv/bin/python
Repository root: /Users/rems/Documents/MScThesis


## 2. Field Declaration

The framework supports the field types used below:

- real scalars,
- complex scalars,
- Dirac fermions,
- gauge bosons,
- ghost fields.

The declarations are explicit about spin, conjugation, indices, and quantum numbers. Those are the ingredients later used by the gauge-aware `Model(...)` workflow.

In [2]:
Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
PhiC = Field(
    "PhiC",
    spin=0,
    self_conjugate=False,
    symbol=S("phiC"),
    conjugate_symbol=S("phiCdag"),
)
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
)
Photon = Field(
    "A",
    spin=1,
    self_conjugate=True,
    symbol=S("A"),
    indices=(LORENTZ_INDEX,),
)
Gluon = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=S("G"),
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)
GhostGluon = Field(
    "ghG",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=S("ghG"),
    conjugate_symbol=S("ghGbar"),
    indices=(COLOR_ADJ_INDEX,),
)

Electron = Field(
    "e",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("e"),
    conjugate_symbol=S("ebar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": Qe},
)
Quark = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
)
W = Field(
    "W",
    spin=1,
    self_conjugate=True,
    symbol=S("W"),
    indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX),
)
B = Field(
    "B",
    spin=1,
    self_conjugate=True,
    symbol=S("B"),
    indices=(LORENTZ_INDEX,),
)
LDoublet = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L"),
    conjugate_symbol=S("Lbar"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": YL},
)
ERight = Field(
    "ER",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("ER"),
    conjugate_symbol=S("ERbar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Y": YR},
)
Higgs = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H"),
    conjugate_symbol=S("Hdag"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": YH},
)

{
    "real scalar": {
        "name": Phi.name,
        "kind": Phi.kind,
        "self_conjugate": Phi.self_conjugate,
        "indices": tuple(index.name for index in Phi.indices),
    },
    "complex scalar": {
        "name": PhiC.name,
        "kind": PhiC.kind,
        "self_conjugate": PhiC.self_conjugate,
        "indices": tuple(index.name for index in PhiC.indices),
    },
    "Dirac fermion": {
        "name": Psi.name,
        "kind": Psi.kind,
        "self_conjugate": Psi.self_conjugate,
        "indices": tuple(index.name for index in Psi.indices),
    },
    "gauge boson": {
        "name": Photon.name,
        "kind": Photon.kind,
        "self_conjugate": Photon.self_conjugate,
        "indices": tuple(index.name for index in Photon.indices),
    },
    "ghost field": {
        "name": GhostGluon.name,
        "kind": GhostGluon.kind,
        "self_conjugate": GhostGluon.self_conjugate,
        "indices": tuple(index.name for index in GhostGluon.indices),
    },
}

{'real scalar': {'name': 'Phi',
  'kind': 'scalar',
  'self_conjugate': True,
  'indices': ()},
 'complex scalar': {'name': 'PhiC',
  'kind': 'scalar',
  'self_conjugate': False,
  'indices': ()},
 'Dirac fermion': {'name': 'Psi',
  'kind': 'fermion',
  'self_conjugate': False,
  'indices': ('Spinor',)},
 'gauge boson': {'name': 'A',
  'kind': 'vector',
  'self_conjugate': True,
  'indices': ('Lorentz',)},
 'ghost field': {'name': 'ghG',
  'kind': 'ghost',
  'self_conjugate': False,
  'indices': ('ColorAdj',)}}

A few points are worth noting immediately:

- complex fields carry both `symbol` and `conjugate_symbol`,
- fermions carry a spinor index,
- gauge bosons carry a Lorentz index and, for non-abelian groups, an adjoint gauge index,
- ghosts are declared explicitly with `kind="ghost"`.

The conjugated version of a non-self-conjugate field is written with `.bar`, for example `Psi.bar`, `Electron.bar`, or `Higgs.bar`.

## 3. Gauge Groups, Representations, and Charges

The logic is:

- attach abelian charges through `quantum_numbers={...}` on the field,
- attach non-abelian structure through field indices plus `GaugeRepresentation(...)`,
- collect the symmetry data into `GaugeGroup(...)` objects,
- then assemble everything into a `Model(...)`.

In [3]:
COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental",
)
WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

U1QED = GaugeGroup(
    name="U1QED",
    abelian=True,
    coupling=e,
    gauge_boson=Photon,
    charge="Q",
)
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gs,
    gauge_boson=Gluon,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU3C_GHOST = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gs,
    gauge_boson=Gluon,
    ghost_field=GhostGluon,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)
U1Y = GaugeGroup(
    name="U1Y",
    abelian=True,
    coupling=g1,
    gauge_boson=B,
    charge="Y",
)

{
    "U1QED": {
        "abelian": U1QED.abelian,
        "coupling": U1QED.coupling,
        "gauge boson": U1QED.gauge_boson.name,
        "charge label": U1QED.charge,
    },
    "SU3C": {
        "abelian": SU3C.abelian,
        "coupling": SU3C.coupling,
        "gauge boson": SU3C.gauge_boson.name,
        "representations": tuple(rep.name for rep in SU3C.representations),
    },
    "SU2L x U1Y": {
        "SU2L boson": SU2L.gauge_boson.name,
        "SU2L representation": tuple(rep.name for rep in SU2L.representations),
        "U1Y boson": U1Y.gauge_boson.name,
        "U1Y charge label": U1Y.charge,
    },
}

{'U1QED': {'abelian': True,
  'coupling': e,
  'gauge boson': 'A',
  'charge label': 'Q'},
 'SU3C': {'abelian': False,
  'coupling': gs,
  'gauge boson': 'G',
  'representations': ('fundamental',)},
 'SU2L x U1Y': {'SU2L boson': 'W',
  'SU2L representation': ('doublet',),
  'U1Y boson': 'B',
  'U1Y charge label': 'Y'}}

In [4]:
{
    "electron charge entry": Electron.quantum_numbers["Q"],
    "quark color representation": SU3C.matter_representation(Quark).name,
    "left doublet weak representation": SU2L.matter_representation(LDoublet).name,
    "left doublet hypercharge entry": LDoublet.quantum_numbers["Y"],
    "Higgs hypercharge entry": Higgs.quantum_numbers["Y"],
}

{'electron charge entry': Qe,
 'quark color representation': 'fundamental',
 'left doublet weak representation': 'doublet',
 'left doublet hypercharge entry': YL,
 'Higgs hypercharge entry': YH}

For abelian groups, the relevant field quantum number is looked up by name, for example `"Q"` or `"Y"`. For non-abelian groups, the action is determined by the field index structure together with the chosen `GaugeRepresentation(...)`.

## 4. Building Models

Two user-facing entry points are available today:

- `Lagrangian(...)` for metadata-free local operators built from fields, `PartialD(...)`, `Gamma(...)`, and tensor factors such as `Metric(...)`,
- `Model(..., lagrangian_decl=...)` for declarations that need model metadata, especially `CovD(...)`, `FieldStrength(...)`, `GaugeFixing(...)`, and `GhostLagrangian(...)`.

In [5]:
local_scalar_lagrangian = Lagrangian(lam4 * Phi * Phi * Phi * Phi)

qed_current_model = Model(
    gauge_groups=(U1QED,),
    fields=(Electron, Photon),
    lagrangian_decl=I * Electron.bar * Gamma(mu) * CovD(Electron, mu),
)

{
    "local source term": str(local_scalar_lagrangian),
    "gauge-aware source term": str(qed_current_model.lagrangian_decl),
}

{'local source term': 'lam4 * Phi * Phi * Phi * Phi',
 'gauge-aware source term': '1\x1b𝑖\x1b * e.bar * Gamma(mu) * CovD(e, mu)'}

The object returned by `model.lagrangian()` is the compiled extraction object. That is the object on which you call `feynman_rule(...)`.

## 5. Lagrangian Usage Patterns

This section is a compact catalog of interactions that are already supported. The examples use `show(...)` for readable output. To avoid repetition, most examples extract one representative vertex; full zero-argument discovery is explored in Section 6.


### Scalar Interaction

A local `\phi^4` interaction can be used directly through `Lagrangian(...)`. Calling `feynman_rule(...)` with explicit fields extracts one vertex; calling it with no fields lists the available signatures.

In [6]:
scalar_lagrangian = Lagrangian(lam4 * Phi * Phi * Phi * Phi)

show(
    "Scalar vertex Gamma(Phi, Phi, Phi, Phi)",
    scalar_lagrangian.feynman_rule(Phi, Phi, Phi, Phi, simplify=True),
)
show(
    "Scalar vertices discovered by feynman_rule()",
    scalar_lagrangian.feynman_rule(simplify=True),
)


Scalar vertex Gamma(Phi, Phi, Phi, Phi)
24𝑖*lam4*(2*𝜋)^d*Delta(q1+q2+q3+q4)

Scalar vertices discovered by feynman_rule()
1 vertex signature(s)

Vertex: ('Phi', 'Phi', 'Phi', 'Phi')
Rule: 24𝑖*lam4*(2*𝜋)^d*Delta(q1+q2+q3+q4)



### Fermion Interaction

A Yukawa-type local term `y \bar\psi \psi \phi` is also handled directly by `Lagrangian(...)`.


In [7]:
fermion_lagrangian = Lagrangian(y * Psi.bar * Psi * Phi)

show(
    "Yukawa vertex Gamma(Psi.bar, Psi, Phi)",
    fermion_lagrangian.feynman_rule(Psi.bar, Psi, Phi, simplify=True),
)


Yukawa vertex Gamma(Psi.bar, Psi, Phi)
1𝑖*y*(2*𝜋)^d*g(bis(4, i1),bis(4, i2))*Delta(q1+q2+q3)



The bispinor metric contracts the two spinor legs. This is the expected momentum-space Yukawa vertex, with the usual overall `i` and delta function.

### Derivative Interaction

Nested `PartialD(...)` factors are supported for local derivative operators. Here the Lagrangian contains both `\phi^4` and `\phi^3 \, \partial^2 \phi`; the two contributions are merged into the same four-scalar vertex.


In [8]:
derivative_lagrangian = Lagrangian(
    lam4 * Phi * Phi * Phi * Phi,
    gBox * Phi * Phi * Phi * PartialD(PartialD(Phi, mu), nu) * Metric(mu, nu),
)

show(
    "Scalar vertex with local and derivative contributions",
    derivative_lagrangian.feynman_rule(Phi, Phi, Phi, Phi, simplify=True),
)


Scalar vertex with local and derivative contributions
24𝑖*lam4*(2*𝜋)^d*Delta(q1+q2+q3+q4)-6𝑖*gBox*(2*𝜋)^d*pcomp(q1,mu)^2*Delta(q1+q2+q3+q4)-6𝑖*gBox*(2*𝜋)^d*pcomp(q2,mu)^2*Delta(q1+q2+q3+q4)-6𝑖*gBox*(2*𝜋)^d*pcomp(q3,mu)^2*Delta(q1+q2+q3+q4)-6𝑖*gBox*(2*𝜋)^d*pcomp(q4,mu)^2*Delta(q1+q2+q3+q4)



The derivative term contributes a sum of `q_1^2 + q_2^2 + q_3^2 + q_4^2`. That is exactly what one expects: in a symmetric `\phi^3 \partial^2 \phi` operator, any one of the four external scalar legs can be the differentiated field.

### Gauge Interaction

Pure gauge sectors are declared with `FieldStrength(...)`. The same compiled Lagrangian can extract one chosen Yang-Mills vertex or enumerate all generated pure-gauge signatures.

In [9]:
yang_mills_model = Model(
    gauge_groups=(SU3C,),
    fields=(Gluon,),
    lagrangian_decl=(
        -(Expression.num(1) / Expression.num(4))
        * FieldStrength(SU3C, mu, nu)
        * FieldStrength(SU3C, mu, nu)
    ),
)
yang_mills_lagrangian = yang_mills_model.lagrangian()

show(
    "Yang-Mills vertex ",
    yang_mills_lagrangian.feynman_rule( simplify=True),
)


Yang-Mills vertex 
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*g(coad(8, i2),coad(8, i4))*pcomp(q1,rho_G_SU3C)*pcomp(q2,rho_G_SU3C)*Delta(q1+q2)-1𝑖*(2*𝜋)^d*g(coad(8, i2),coad(8, i4))*pcomp(q1,i3)*pcomp(q2,i1)*Delta(q1+q2)

Vertex: ('G', 'G', 'G')
Rule: -gs*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q1,i5)*Delta(q1+q2+q3)+gs*(2*𝜋)^d*g(mink(4, i1),mink(4, i3))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q2,i5)*Delta(q1+q2+q3)+gs*(2*𝜋)^d*g(mink(4, i1),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q1,i3)*Delta(q1+q2+q3)-gs*(2*𝜋)^d*g(mink(4, i1),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q3,i3)*Delta(q1+q2+q3)-gs*(2*𝜋)^d*g(mink(4, i3),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q2,i1)*Delta(q1+q2+q3)+gs*(2*𝜋)^d*g(mink(4, i3),mink(4, i5))*f(coad(8, i2),coad(8, i4),coad(8, i6))*pcomp(q3,i1)*Delta(q1+q2+q3)

Vertex: ('G', 'G', 'G', 'G')
Rule: 1𝑖*gs^2*(2*𝜋)^d*g(mink(4, i1),mi

This output is deliberately explicit: Lorentz metrics, momenta, and the non-abelian structure constant are all shown. It is the expected three-gluon Yang-Mills vertex written in the codebase's current symbolic conventions.

### Ghost Interaction

The non-abelian ghost sector is exposed through `GhostLagrangian(...)` once the gauge group knows its ghost field.

In [10]:
ghost_model = Model(
    gauge_groups=(SU3C_GHOST,),
    fields=(Gluon, GhostGluon),
    lagrangian_decl=GhostLagrangian(SU3C_GHOST),
)
ghost_lagrangian = ghost_model.lagrangian()

show(
    "Ghost-gluon vertex Gamma(ghG.bar, G, ghG)",
    ghost_lagrangian.feynman_rule(GhostGluon.bar, Gluon, GhostGluon, simplify=True),
)


Ghost-gluon vertex Gamma(ghG.bar, G, ghG)
-gs*(2*𝜋)^d*f(coad(8, i1),coad(8, i3),coad(8, i4))*pcomp(q1,i2)*Delta(q1+q2+q3)



The cubic antighost-gluon-ghost vertex carries the antighost momentum. That is the correct momentum dependence for the ordinary Faddeev-Popov ghost term in integrated form.

### QED

For abelian gauge interactions, one declares the charge on the field and writes the kinetic term with `CovD(...)`.

In [11]:
qed_model = Model(
    gauge_groups=(U1QED,),
    fields=(Electron, Photon),
    lagrangian_decl=I * Electron.bar * Gamma(mu) * CovD(Electron, mu),
)
qed_lagrangian = qed_model.lagrangian()

show(
    "QED vertex Gamma(e.bar, e, A)",
    qed_lagrangian.feynman_rule(Electron.bar, Electron, Photon, simplify=True),
)


QED vertex Gamma(e.bar, e, A)
-1𝑖*e*Qe*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, i3))*Delta(q1+q2+q3)



This is the expected abelian current vertex: a gamma matrix times the charge factor `Qe` and the coupling `e`. The sign convention follows the codebase's covariant-derivative conventions.

### QCD

For non-abelian matter, the field carries a color index and the gauge group supplies the generator structure.

In [12]:
qcd_model = Model(
    gauge_groups=(SU3C,),
    fields=(Quark, Gluon),
    lagrangian_decl=I * Quark.bar * Gamma(mu) * CovD(Quark, mu),
)
qcd_lagrangian = qcd_model.lagrangian()

show(
    "QCD quark-gluon vertex Gamma(q.bar, q, G)",
    qcd_lagrangian.feynman_rule(Quark.bar, Quark, Gluon, simplify=True),
)


QCD quark-gluon vertex Gamma(q.bar, q, G)
-1𝑖*gs*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i3),mink(4, i5))*t(coad(8, i6),cof(3, i2),cof(3, i4))*Delta(q1+q2+q3)



Compared with QED, the new ingredient is the generator `t(...)` acting on the color indices. That is the standard non-abelian color structure of the quark-gluon vertex.

### Full SU(2) x U(1) Electroweak / SM-Style Sector

The next example combines a left-handed lepton doublet, a right-handed singlet, and a Higgs doublet in one unbroken electroweak model. This is a realistic SM-style slice of the supported API.

In [13]:
electroweak_model = Model(
    gauge_groups=(SU2L, U1Y),
    fields=(LDoublet, ERight, Higgs, W, B),
    lagrangian_decl=(
        I * LDoublet.bar * Gamma(mu) * CovD(LDoublet, mu)
        + I * ERight.bar * Gamma(mu) * CovD(ERight, mu)
        + CovD(Higgs.bar, mu) * CovD(Higgs, mu)
    ),
)
electroweak_lagrangian = electroweak_model.lagrangian()

print(
    "Electroweak current Gamma(L.bar, L, W)\n",
    electroweak_lagrangian.feynman_rule(LDoublet.bar, LDoublet, W, simplify=True),
)
print(
    "Hypercharge current Gamma(ER.bar, ER, B)\n ",
    electroweak_lagrangian.feynman_rule(ERight.bar, ERight, B, simplify=True),
)
print(
    "Mixed Higgs contact Gamma(H.bar, H, W, B)  \n",
    electroweak_lagrangian.feynman_rule(Higgs.bar, Higgs, W, B, simplify=True),
)

print("=============================================================")

show(
    "Electroweak",
    electroweak_lagrangian.feynman_rule( simplify=True),
)

Electroweak current Gamma(L.bar, L, W)
 -1𝑖*g2*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i3),mink(4, i5))*t(coad(3, i6),cof(2, i2),cof(2, i4))*Delta(q1+q2+q3)
Hypercharge current Gamma(ER.bar, ER, B)
  -1𝑖*g1*YR*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, i3))*Delta(q1+q2+q3)
Mixed Higgs contact Gamma(H.bar, H, W, B)  
 2𝑖*g1*g2*YH*(2*𝜋)^d*g(mink(4, i3),mink(4, i5))*t(coad(3, i4),cof(2, i1),cof(2, i2))*Delta(q1+q2+q3+q4)
Electroweak
11 vertex signature(s)

Vertex: ('L.bar', 'L', 'W')
Rule: -1𝑖*g2*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i3),mink(4, i5))*t(coad(3, i6),cof(2, i2),cof(2, i4))*Delta(q1+q2+q3)

Vertex: ('L.bar', 'L', 'B')
Rule: -1𝑖*g1*YL*(2*𝜋)^d*g(cof(2, i2),cof(2, i4))*gamma(bis(4, i1),bis(4, i3),mink(4, i5))*Delta(q1+q2+q3)

Vertex: ('L.bar', 'L')
Rule: 1𝑖*(2*𝜋)^d*g(cof(2, i2),cof(2, i4))*gamma(bis(4, i1),bis(4, i3),mink(4, mu))*pcomp(q2,mu)*Delta(q1+q2)

Vertex: ('ER.bar', 'ER', 'B')
Rule: -1𝑖*g1*YR*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, i3))*Delta(q1+q2+q3)

Vertex: ('ER.bar', 'ER')


This one model already shows several distinct capabilities:

- the SU(2) current of a weak doublet,
- the U(1) hypercharge current of a singlet,
- the mixed Higgs-gauge contact term from `(D_\mu H)^\dagger (D^\mu H)`.

That is exactly the kind of electroweak structure one expects from the implemented `CovD(...)` lowering.

## 6. Vertices, Feynman Rules, and Discovery

`feynman_rule(...)` has two user-facing modes:

- with explicit fields, it returns one momentum-space vertex expression;
- with no fields, it returns a dictionary mapping vertex signatures to expressions.

The output expressions contain coupling constants, momentum dependence, open index structure, and the overall momentum-conservation factor. The order of explicit fields fixes the displayed momenta and automatically assigned open indices.

In [14]:
p_in, p_out, k = S("p_in", "p_out", "k")

show(
    "QED vertex with default momenta",
    qed_lagrangian.feynman_rule(Electron.bar, Electron, Photon, simplify=True),
)
show(
    "QED vertex with custom momenta",
    qed_lagrangian.feynman_rule(
        Electron.bar,
        Electron,
        Photon,
        momenta=[p_in, p_out, k],
        simplify=True,
    ),
)


QED vertex with default momenta
-1𝑖*e*Qe*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, i3))*Delta(q1+q2+q3)

QED vertex with custom momenta
-1𝑖*e*Qe*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, i3))*Delta(p_in+p_out+k)



The structure is easy to read once the conventions are clear:

- `Delta(...)` is the overall momentum-conservation factor,
- `gamma(...)` is the open gamma-matrix structure,
- `t(...)` and `f(...)` are generator and structure-constant tensors when non-abelian groups are involved,
- `g(...)` is the metric tensor,
- `pcomp(p, mu)` is the component of momentum `p` along index `mu`.

Using `momenta=[...]` is useful when you want the displayed momentum labels to match a paper or calculation.

### Enumerating Vertices in a Larger Gauge Sector

Calling `feynman_rule()` with no field arguments asks the compiled Lagrangian for every available vertex signature. By default, the result is a dictionary keyed by field-name tuples, for example `("ghW.bar", "W", "ghW")`. If you need the actual `Field` / `Field.bar` objects as keys, use `key_format="fields"`.

For a larger model, use `simplify=False` first so discovery stays lightweight. The `show(...)` helper below prints each discovered signature followed by the corresponding expression.

The model below contains SU(2) x U(1) matter covariant derivatives, both field-strength terms, both gauge-fixing terms, and the non-abelian SU(2) ghost sector. The abelian U(1) gauge-fixing term is included; the ordinary ghost declaration is only used for SU(2), where the non-abelian ghost-gauge coupling is nontrivial.

In [15]:
WeakGhost = Field(
    "ghW",
    spin=0,
    kind="ghost",
    self_conjugate=False,
    symbol=S("ghW"),
    conjugate_symbol=S("ghWbar"),
    indices=(WEAK_ADJ_INDEX,),
)

SU2L_GHOST = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    ghost_field=WeakGhost,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)

ew_gauge_fixed_model = Model(
    gauge_groups=(SU2L_GHOST, U1Y),
    fields=(LDoublet, ERight, Higgs, W, B, WeakGhost),
    lagrangian_decl=(
        I * LDoublet.bar * Gamma(mu) * CovD(LDoublet, mu)
        + I * ERight.bar * Gamma(mu) * CovD(ERight, mu)
        + CovD(Higgs.bar, mu) * CovD(Higgs, mu)
        + (
            -(Expression.num(1) / Expression.num(4))
            * FieldStrength(SU2L_GHOST, mu, nu)
            * FieldStrength(SU2L_GHOST, mu, nu)
        )
        + (
            -(Expression.num(1) / Expression.num(4))
            * FieldStrength(U1Y, mu, nu)
            * FieldStrength(U1Y, mu, nu)
        )
        + GaugeFixing(SU2L_GHOST, xi=xi)
        + GaugeFixing(U1Y, xi=xi)
        + GhostLagrangian(SU2L_GHOST)
    ),
)

ew_gauge_fixed_lagrangian = ew_gauge_fixed_model.lagrangian()
ew_gauge_fixed_rules = ew_gauge_fixed_lagrangian.feynman_rule(simplify=False)

print("Compiled interaction terms:", len(ew_gauge_fixed_lagrangian.terms))
show(
    "Gauge-fixed SU(2) x U(1) vertices discovered by feynman_rule()",
    ew_gauge_fixed_rules,
)

ew_gauge_fixed_rules_by_field = ew_gauge_fixed_lagrangian.feynman_rule(
    simplify=False,
    key_format="fields",
)
print("First object-keyed signature:", next(iter(ew_gauge_fixed_rules_by_field)))


Compiled interaction terms: 24
Gauge-fixed SU(2) x U(1) vertices discovered by feynman_rule()
17 vertex signature(s)

Vertex: ('L.bar', 'L', 'W')
Rule: -1𝑖*g2*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i3),mink(4, i5))*t(coad(3, i6),cof(2, i2),cof(2, i4))*delta(W,W)*delta(L,L)*delta(Lbar,Lbar)*Delta(q1+q2+q3)

Vertex: ('L.bar', 'L', 'B')
Rule: -1𝑖*g1*YL*(2*𝜋)^d*g(cof(2, i2),cof(2, i4))*gamma(bis(4, i1),bis(4, i3),mink(4, i5))*delta(B,B)*delta(L,L)*delta(Lbar,Lbar)*Delta(q1+q2+q3)

Vertex: ('L.bar', 'L')
Rule: 1𝑖*(2*𝜋)^d*g(cof(2, i2),cof(2, i4))*gamma(bis(4, i1),bis(4, i3),mink(4, mu))*pcomp(q2,mu)*delta(L,L)*delta(Lbar,Lbar)*Delta(q1+q2)

Vertex: ('ER.bar', 'ER', 'B')
Rule: -1𝑖*g1*YR*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, i3))*delta(B,B)*delta(ER,ER)*delta(ERbar,ERbar)*Delta(q1+q2+q3)

Vertex: ('ER.bar', 'ER')
Rule: 1𝑖*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, mu))*pcomp(q2,mu)*delta(ER,ER)*delta(ERbar,ERbar)*Delta(q1+q2)

Vertex: ('H.bar', 'H', 'W')
Rule: -1𝑖*g2*(2*𝜋)^d*t(coad(3, i4),cof(

## 7. Capability Map

The notebook demonstrates the following current capabilities:

- field declarations for real scalars, complex scalars, Dirac fermions, gauge bosons, and non-abelian ghosts;
- local polynomial interactions, Yukawa-like fermion couplings, and derivative operators built from `PartialD(...)`;
- abelian and non-abelian gauge currents via `CovD(...)`;
- pure gauge sectors via `FieldStrength(...)`, including Yang-Mills cubic and quartic self-interactions;
- gauge fixing via `GaugeFixing(...)` and ordinary non-abelian Faddeev-Popov ghosts via `GhostLagrangian(...)`;
- mixed gauge sectors such as SU(2) x U(1), including matter currents and scalar gauge-contact terms;
- targeted vertex extraction through `feynman_rule(Field1, Field2, ...)`;
- full vertex discovery through zero-argument `feynman_rule()`, keyed by readable field-name tuples by default;
- object-keyed discovery through `feynman_rule(key_format="fields")` when programmatic lookup by `Field` objects is needed.

A practical rule of thumb is: use explicit fields when you know the process, and use zero-argument `feynman_rule()` when you want to inspect what the compiled Lagrangian contains.

## 8. Practical Notes and Conventions

- Use `Field.bar` for conjugated non-self-conjugate fields in both declarations and explicit `feynman_rule(...)` calls.
- Use `Lagrangian(...)` for local terms that do not require gauge metadata.
- Use `Model(..., lagrangian_decl=...)` when the declaration involves `CovD(...)`, `FieldStrength(...)`, `GaugeFixing(...)`, or `GhostLagrangian(...)`.
- Default external momenta are named `q1`, `q2`, `q3`, ... in the order you pass the external fields.
- By default the returned expression includes the universal `(2*pi)^d Delta(sum p)` factor.
- The order of fields in explicit `feynman_rule(...)` calls fixes the displayed momentum labels and open indices, so use a consistent ordering when comparing vertices.
- For non-abelian ghosts, the gauge group must know its ghost field through `ghost_field=...`.
- If a field carries the same index type more than once, you may need `GaugeRepresentation(slot=...)` or `slot_policy='sum'` to resolve the intended action.
- Zero-argument `feynman_rule()` returns name-tuple keys such as `("ghW.bar", "W", "ghW")`; use `key_format="fields"` if you need `Field` / `Field.bar` objects as keys.
- For large models, start with `simplify=False` when enumerating all vertices, then extract important vertices explicitly with `simplify=True`.


## 9. Conclusion

With the API demonstrated here, a user can define fields and gauge groups, write local and gauge-aware Lagrangians, compile QED-, QCD-, ghost-, and electroweak-style sectors, and extract symbolic momentum-space vertices.

The current workflow now supports both targeted calculations and broad inspection: call `feynman_rule(...)` with explicit fields for one process, or call zero-argument `feynman_rule()` to see the available vertices in a compiled Lagrangian.
